# Hybrid Search (BM25 + Dense for retrieval & RRF for top k)

<img src="../images/" width=“500”>

In [2]:
from langchain_core.globals import set_debug
set_debug(False)

## Initialize Azure Chat OpenAI

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv(override=True, verbose=True)

chat_llm = AzureChatOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
)

## Initialize Azure Embedding Open AI

In [4]:
emd_llm = AzureOpenAIEmbeddings(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
)

## Load the Document and upload it to vector store

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader

file = '../Data/HTTP-Status-Codes.pdf'
pdf_docs = PyMuPDFLoader(file_path=file).load()


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents=pdf_docs)

In [7]:
from langchain_community.vectorstores import FAISS

faiss = FAISS.from_documents(documents=chunks, embedding=emd_llm)


## User question 

In [8]:
num_queries = 3
user_question = "What is error code 404?"

## Rewrite the Query using LLM

In [9]:
REWRITE_QUERY_PROMPT = """"
You rewrite a user's question into a single, clear, self-contained search query optimized for retrieving relevant documents from a vector database. Resolve vague references, add implied specifics, and remove conversational filler. Output ONLY  the rewritten query - no preamble, no quotes, one line.
"""

In [10]:
from langchain_core.prompts import ChatPromptTemplate
rewrite_query_prompt_template = ChatPromptTemplate.from_messages(
    [
        ('system', REWRITE_QUERY_PROMPT),
        ('user','{user_question}')
    ]
).invoke({'user_question':user_question})

In [11]:
llm_response = chat_llm.invoke(rewrite_query_prompt_template)

In [12]:
print(llm_response)

print(llm_response.content)

content='HTTP 404 Not Found status code definition, causes, how it works, common scenarios and examples, debugging and troubleshooting steps to fix 404 errors (client vs server causes), and differences from other 4xx codes like 410/403' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 250, 'prompt_tokens': 73, 'total_tokens': 323, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 2198, 'pre_inference_ms': 89, 'service_tbt_ms': 7, 'service_ttft_ms': 493, 'service_ttlt_ms': 2671, 'total_duration_ms': 2592, 'user_visible_ttft_ms': 404}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dx9rBhi0D9JROCFLy6QrnssxiI0rr', 'service_tier': 'default', 'promp

## Dense retrieval

In [13]:

modified_query_by_llm = llm_response.content
faiss_chunks = faiss.similarity_search_with_score(query=modified_query_by_llm, k=num_queries)

In [14]:
print(f"Chunks fetched from FAISS Vector store: {len(faiss_chunks)}")
faiss_top_chunks = []
for c, score in faiss_chunks:
    print(score)
    print(c.page_content, end="\n-----\n")
    
    faiss_top_chunks.append(c.page_content)

Chunks fetched from FAISS Vector store: 3
0.7885672
404 Not Found Error Impact
410 Gone vs 404 Error Codes
500 Internal Server Error Solutions
503 Service Unavailable Fix
Common HTTP Status Code FAQ
Search...
Tutorials
Comparisons
News
Keep in touch
-----
0.81075656
200 OK when it works or 404 Not Found when the resource doesn’t exist.
How to Check HTTP Status Code Using Tools
You need to see status codes when debugging. Several tools make this easy.
-----
0.9453822
permanent.
4XX codes flag client errors. You typed a wrong URL. You don’t have
permission. The page doesn’t exist. These are on you or your users.
-----


In [15]:
from langchain_core.prompts import ChatPromptTemplate
PROMPT_TEMPLATE = """"
You are a helpful assistance. You answer user's query {question} from the provided context {context}. If context isn't sufficient to provide the answer, inform the user that Context isn't sufficient to provide the answer of your query.
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT_TEMPLATE),
        ("user",  "user query: {question}"),
    ]
)


In [16]:
prompt =  prompt_template.invoke({'question': user_question, 'context': user_question})
response = chat_llm.invoke(prompt)
print(response)

print(response.content)

content='A 404 (Not Found) is an HTTP status code meaning the server was reached but cannot find the requested resource (the URL you asked for). It’s a client‑side error in the 4xx range and typically means the page or file was moved or deleted, or the URL was typed or linked incorrectly.\n\nCommon reasons:\n- Typo in the URL\n- The page was removed or moved without a redirect\n- Broken link from another site\n- Caching serving an outdated link\n\nWhat you can do as a user:\n- Check the URL for typos\n- Try the site’s homepage or search function\n- Refresh the page or clear your browser cache\n- Contact the site owner if the page should exist\n\nWhat a site owner/developer can do:\n- Restore the resource or add a proper redirect (e.g., 301) to the new location\n- Provide a helpful custom 404 page with navigation/search\n- Fix broken links and update sitemap/links\n- Monitor logs to find frequent 404s and address them\n\nIf you want, tell me the exact URL or where you saw the 404 and I 